In [ ]:
import re
import json

import pandas as pd

from google import genai
from google.genai import types

from utils import load_research_questions
from prompts import SYSTEM_PROMPT, USER_PROMPT


client = genai.Client()

In [ ]:
questions = load_research_questions(
    "../batches/batch_suggestions_results/GT1000batch_6932ad05f04081908c4727f8a46aa838_output.jsonl"
)

In [ ]:
messages = []
for custom_id, rq in questions:
    req = {
        "systemInstruction": {
            "parts": [
                {
                    "text": SYSTEM_PROMPT.format(research_question=rq),
                }
            ]
        },
        "contents": [
            {
                "role": "user",
                "parts": [
                    {
                        "text": USER_PROMPT,
                    }
                ]
            }
        ],
    }

    line = {"key": f"{custom_id}", "request": req}
    messages.append(line)

# with open("gemini_suggestions_input.jsonl", 'w', encoding='utf-8') as f:
#     for item in messages:
#         json.dump(item, f, ensure_ascii=False)
#         f.write('\n')

In [ ]:
# uploaded = client.files.upload(
#     file="gemini_suggestions_input.jsonl",
#     config=types.UploadFileConfig(mime_type="jsonl"),
# )

# job = client.batches.create(
#     model="gemini-3-pro-preview",
#     src=uploaded.name,
# )

In [ ]:
# retrieving
# check status of all batches
batches = client.batches.list()
for batch in batches:
    print(batch.name, batch.state)

In [ ]:
# BATCH_ID = "..."

# client.batches.get(name=f"batches/{BATCH_ID}")
# file_content = client.files.download(file=f"files/batch-{BATCH_ID}")

# with open("gemini_suggestions_output.jsonl", "wb") as f:
#     f.write(file_content)

In [ ]:
df = pd.read_json("gemini_suggestions_output.jsonl", lines=True)

In [ ]:
def extract_text(x):
    try:
        return x["candidates"][0]["content"]["parts"][-1]["text"]
    except (TypeError, KeyError, IndexError):
        return None

df["text"] = df.response.apply(extract_text)

import json

df["text"] = (
    df["text"]
      .str.strip()
      .str.replace(r"^```(?:json)?\s*|\s*```$", "", regex=True, case=False)
      .apply(lambda s: json.loads(s) if isinstance(s, str) else None)
)

In [ ]:
df["text"][0]